In [ ]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path().resolve().parent.parent / "data" / "raw" / "finanças"
OUTPUT_DATA_PATH = (
    Path().resolve().parent.parent / "data" / "processed"
)

COLS_COMUNS = [
    "ano_eleicao", "sg_uf", "ds_cargo", "sg_partido", "nr_candidato",
    "ds_origem_receita", "ds_fonte_receita", "dt_receita", "vr_receita",
]


# Receitas 2018/2022 (formato pós-FEFC)
def importar_receitas(path_and_file_receitas: str) -> pd.DataFrame:
    with open(path_and_file_receitas, encoding="latin1") as file:
        receitas = pd.read_csv(
            file,
            sep=";",
            dtype={"CD_ESFERA_PARTIDARIA_DOADOR": "str", "NR_DOCUMENTO_DOACAO": "str"},
        )
    receitas.columns = receitas.columns.str.lower()
    receitas["ds_cargo"] = receitas["ds_cargo"].str.upper()
    receitas["vr_receita"] = receitas["vr_receita"].str.replace(",", ".").astype(float)
    receitas["nr_candidato"] = receitas["nr_candidato"].astype(str)
    return receitas


# Receitas 2014 (formato pré-FEFC — colunas diferentes)
def importar_receitas_2014(path_and_file) -> pd.DataFrame:
    rename_map = {
        "uf": "sg_uf",
        "sigla  partido": "sg_partido",   # duplo espaço preservado do original
        "numero candidato": "nr_candidato",
        "cargo": "ds_cargo",
        "data da receita": "dt_receita",
        "valor receita": "vr_receita",
        "tipo receita": "ds_origem_receita",
        "fonte recurso": "ds_fonte_receita",
    }
    df = pd.read_csv(path_and_file, sep=";", encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns=rename_map)
    df["ds_cargo"] = df["ds_cargo"].str.upper()
    df["nr_candidato"] = df["nr_candidato"].astype(str)
    df["sg_partido"] = df["sg_partido"].astype(str)
    df["sg_uf"] = df["sg_uf"].astype(str)
    # Formato sem espaço: "02/10/201400:00:00" → extrair os 10 primeiros chars
    df["dt_receita"] = df["dt_receita"].str[:10]
    df["vr_receita"] = df["vr_receita"].astype(str).str.replace(",", ".").astype(float)
    df["ano_eleicao"] = 2014
    return df


def processar_receitas() -> pd.DataFrame:
    receitas_2014 = importar_receitas_2014(
        RAW_DATA_PATH / "receitas_candidatos_2014_brasil.txt"
    )
    receitas_2018 = importar_receitas(
        RAW_DATA_PATH / "receitas_candidatos_2018_BRASIL.csv"
    )
    receitas_2022 = importar_receitas(
        RAW_DATA_PATH / "receitas_candidatos_2022_BRASIL.csv"
    )

    receitas = pd.concat(
        [
            receitas_2014[COLS_COMUNS],
            receitas_2018[COLS_COMUNS],
            receitas_2022[COLS_COMUNS],
        ],
        axis=0,
        ignore_index=True,
    )

    receitas = receitas.groupby(
        [
            "ano_eleicao",
            "sg_uf",
            "ds_cargo",
            "sg_partido",
            "nr_candidato",
            "ds_origem_receita",
            "ds_fonte_receita",
            "dt_receita"
        ],
        as_index=False,
    ).agg({"vr_receita": "sum"})

    receitas['dt_receita'] = pd.to_datetime(receitas['dt_receita'], format='%d/%m/%Y')

    receitas.to_parquet(OUTPUT_DATA_PATH / "receitas.parquet", index=False)

    return receitas


receitas = processar_receitas()
print(receitas.groupby("ano_eleicao")["vr_receita"].count())


In [2]:
# receitas_2022 = importar_receitas(
#         RAW_DATA_PATH / "receitas_candidatos_2022_BRASIL.csv"
#     )

# receitas_2022['dt_receita'] = pd.to_datetime(receitas_2022['dt_receita'], dayfirst=True)
# receitas_2022['mes_receita'] = receitas_2022['dt_receita'].dt.month

# receitas_2022['vr_receita'] = receitas_2022['vr_receita'] / 1_000_000

# plot = receitas_2022.groupby('dt_receita')['vr_receita'].sum().reset_index()

# receitas_2022.groupby('mes_receita')['vr_receita'].sum().reset_index()


# import plotly.express as px

# plot = receitas_2022.groupby('dt_receita')['vr_receita'].sum().reset_index()

# px.line(plot, x='dt_receita', y='vr_receita')